# Notebook 04 -- Data Quality Screening

**Thesis:** Content Moderation on TikTok and X -- A Comparative Analysis of DSA Statements of Reasons (2025)  
**Author:** Mohsen Zahedi  

> **Note:** This notebook is pre-run. Data lives at `C:\BA_Data\` and is not included in this repository. The full screening report is available at `02_data_cleaning/logs/pre_harmonized_screening_report.md`.

---

This notebook presents the data quality check run on the pre-harmonized working files before analysis. It covers completeness (missing values), uniqueness, and any data quality flags that informed analytical decisions.

**Script:** `02_data_cleaning/scripts/screen_pre_harmonized_data.py`

In [1]:
from pathlib import Path

import polars as pl

DATA_ROOT = Path(r"C:\BA_Data")
TIKTOK_FILE = DATA_ROOT / "tiktok_de_2025.parquet"
X_FILE = DATA_ROOT / "x_de_2025.parquet"

## 1. Row and Column Counts

In [2]:
for label, file in [("TikTok", TIKTOK_FILE), ("X", X_FILE)]:
    schema = pl.read_parquet_schema(str(file))
    rows = pl.scan_parquet(str(file)).select(pl.len()).collect().item()
    print(f"{label}: {rows:,} rows, {len(schema)} columns")

TikTok: 997,688,267 rows, 10 columns
X: 183,321 rows, 10 columns


## 2. Completeness -- Missing Values per Column

Missing values reduce analytical coverage. Any column with significant missingness needs to be flagged before use in the analysis.

In [3]:
def null_report(file: Path, label: str) -> pl.DataFrame:
    schema = pl.read_parquet_schema(str(file))
    total_rows = pl.scan_parquet(str(file)).select(pl.len()).collect().item()
    null_exprs = [pl.col(c).null_count().alias(c) for c in schema]
    null_counts = pl.scan_parquet(str(file)).select(null_exprs).collect(engine="streaming").to_dicts()[0]
    rows = [
        {
            "column": col,
            "null_count": null_counts[col],
            "null_pct": f"{null_counts[col] / total_rows * 100:.2f}%",
        }
        for col in schema
    ]
    print(f"\n{label} ({total_rows:,} rows):")
    return pl.DataFrame(rows)

null_report(TIKTOK_FILE, "TikTok")


TikTok (997,688,267 rows):


column,null_count,null_pct
str,i64,str
"""uuid""",0,"""0.00%"""
"""category""",0,"""0.00%"""
"""content_type""",0,"""0.00%"""
"""automated_detection""",0,"""0.00%"""
"""automated_decision""",0,"""0.00%"""
"""territorial_scope""",0,"""0.00%"""
"""application_date""",0,"""0.00%"""
"""platform_name""",0,"""0.00%"""
"""decision_visibility""",12379074,"""1.24%"""


In [4]:
null_report(X_FILE, "X")


X (183,321 rows):


column,null_count,null_pct
str,i64,str
"""uuid""",0,"""0.00%"""
"""category""",0,"""0.00%"""
"""content_type""",0,"""0.00%"""
"""automated_detection""",0,"""0.00%"""
"""automated_decision""",0,"""0.00%"""
"""territorial_scope""",0,"""0.00%"""
"""application_date""",0,"""0.00%"""
"""platform_name""",0,"""0.00%"""
"""decision_visibility""",0,"""0.00%"""


### Finding: TikTok `decision_visibility` has 1.24% missing values

12,379,074 TikTok records (1.24%) have no value in `decision_visibility`. This is the only column with missingness in either dataset.

**Implication for analysis:** If the research questions use `decision_visibility` as a variable, these 12.4M records will be excluded from that specific calculation. For category and automation-rate analyses, these records are still fully usable since all other columns are complete.

## 3. Uniqueness -- UUID Column

The `uuid` column should uniquely identify each SoR record. Near-100% uniqueness is expected.

In [5]:
# approx_n_unique uses HyperLogLog -- fast but approximate (+/- ~1.6% standard error)
for label, file in [("TikTok", TIKTOK_FILE), ("X", X_FILE)]:
    result = (
        pl.scan_parquet(str(file))
        .select([
            pl.len().alias("total_rows"),
            pl.col("uuid").approx_n_unique().alias("approx_unique_uuids"),
        ])
        .collect(engine="streaming")
    )
    total = result["total_rows"][0]
    unique = result["approx_unique_uuids"][0]
    print(f"{label}: {total:,} rows | ~{unique:,} unique UUIDs | ~{unique/total*100:.1f}%")

TikTok: 997,688,267 rows | ~942,862,968 unique UUIDs | ~94.5%
X: 183,321 rows | ~189,501 unique UUIDs | ~103.4%


### Interpreting the UUID Results

**X (result > 100%):** The approximate unique count exceeded the total row count. This is a known artifact of HyperLogLog estimation on small datasets (183K rows). The true uniqueness is effectively ~100% -- X has one record per UUID.

**TikTok (~94.5%):** The gap between total rows (997.7M) and approximate unique UUIDs (~942.9M) is approximately 54.8M records. HyperLogLog's expected error margin at this scale is ~16M rows (1.6%). The gap is roughly 3.4x the expected error, which suggests TikTok's dataset contains genuine duplicate UUIDs at scale.

**Impact on this thesis:** UUID is not used as a join key or aggregation identifier in the analysis. All research questions operate on row-level category and automation attributes, so duplicate UUIDs do not affect the results. This is noted here for transparency.

## 4. Low-Cardinality Columns

Columns with very few distinct values define the analytical dimensions of the dataset.

In [6]:
# Show unique values for key categorical columns (X is small enough to collect fully)
categorical_cols = ["content_type", "automated_detection", "automated_decision", "decision_visibility", "api_version"]

for col in categorical_cols:
    x_vals = (
        pl.scan_parquet(str(X_FILE))
        .select(pl.col(col).unique())
        .collect()
        .to_series()
        .to_list()
    )
    print(f"{col}: {sorted(str(v) for v in x_vals)}")

content_type: ['["CONTENT_TYPE_AUDIO"]', '["CONTENT_TYPE_SYNTHETIC_MEDIA"]']
automated_detection: ['No']
automated_decision: ['AUTOMATED_DECISION_FULLY', 'AUTOMATED_DECISION_NOT_AUTOMATED']
decision_visibility: ['["DECISION_VISIBILITY_OTHER"]']
api_version: ['v1', 'v2']


## 5. Summary -- Data Quality Assessment

| Check | TikTok | X | Impact |
|---|---|---|---|
| Missing values | `decision_visibility`: 1.24% | None | Low -- affects only visibility analysis |
| UUID uniqueness | ~94.5% (possible duplicates) | ~100% | None -- UUID not used in analysis |
| Date coverage | 2025-01-01 to 2025-12-31 | 2025-01-01 to 2025-12-31 | Confirmed in scope |
| Schema consistency | 10 columns, all present | 10 columns, all present | No schema issues |
| Unmapped categories | 0 | 0 | Harmonization complete |

**Conclusion:** The working files are ready for analysis. The one active data quality flag (`decision_visibility` missingness on TikTok) will be noted in the methodology chapter when this column is used.